##Installing Azure SDK

In [1]:
import subprocess #imports module which allows running os commands from py
subprocess.run(["pip", "install", "azure-storage-blob==12.19.0"], check=True) #installs py sdk to communicate with Azure Blob Storage v12.19.0 -> newer versions aren't compatible with azurite

CompletedProcess(args=['pip', 'install', 'azure-storage-blob==12.19.0'], returncode=0)

##Creating SparkSession

In [5]:
from pyspark.sql import SparkSession#imports the SparkSession class

spark = SparkSession.builder\
    .appName("bronze_explorer")\
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.20.0")\
    .config("spark.hadoop.fs.azure.account.auth.type.devstoreaccount1.blob.core.windows.net", "SharedKey")\
    .config("spark.hadoop.fs.azure.account.key.devstoreaccount1.blob.core.windows.net", "Eby8vdM02xNOcqFlqUwJPLlmEtlCDXJ1OUzFT50uSRZ6IFsuFq2UVErCz4I6tq/K1SZFPTOtr/KBHBeksoGMGw==")\
    .config("spark.hadoop.fs.azure.storage.emulator.enabled", "true")\
    .config("spark.hadoop.fs.azure.storage.emulator.account.name", "devstoreaccount1")\
    .config("spark.hadoop.fs.azure.storage.emulator.rest.endpoint", "http://azurite:10000")\
    .getOrCreate()

#initiates SparkSession instantiation using the Builder pattern. Below is the configuration setup
#job name for Spark logs
#downloads 2 Java packages (JARs) needed for Azure: the hadoop-azure connector and azure-storage-blob (Azure's Java SDK). PS: hadoop-azure version has to be in the same version as the container's internal Hadoop
#authentication method for devstoreaccount1 - SharedKey is the simplest method, which is used by Azurite
#the authentication key itself for devstoreaccount1 - this is the standard Azurite key found inside the container
#this flag tells Spark it is dealing with a local emulator and not the real Azure
#emulator's account name - always devstoreaccount1 for Azurite
#the adress where Azurite is running - azurite is the name of the docker service and 10000 is the blob storage port
#creates the Spark Session with the settings above

print(spark.version)

3.5.0


## Downloading table (parquet) files from Azurite

In [7]:
from azure.storage.blob import BlobServiceClient#imports Azure SDK main class for blob storage interaction
import os#imports this py module for os interactions such as creating folders and path manipulation

client = BlobServiceClient(
    account_url="http://azurite:10000/devstoreaccount1",#creates client reaching Azurite - azurite is the docker hostname, 10000 the port and devstoreaccount1 the account name
    credential={
        "account_name": "devstoreaccount1",
        "account_key": "Eby8vdM02xNOcqFlqUwJPLlmEtlCDXJ1OUzFT50uSRZ6IFsuFq2UVErCz4I6tq/K1SZFPTOtr/KBHBeksoGMGw=="})#passing Azurite credentials as a py dictionaryt

container = client.get_container_client("bronze")#points client to bronze container which is equivalent to a lake root folder
os.makedirs("/tmp/bronze/gdp_raw", exist_ok=True)#creates a temporary bronze folder for the table in the container. if it already exists: don't fail the run

for blob in container.list_blobs(name_starts_with="gdp_raw/part-"):#lists all files inside the bronze container. filters only real parquet files, leaving out Spark control files 
    blob_client = container.get_blob_client(blob)#creates a specific client for each file individually - needed for download
    with open(f"/tmp/bronze/{blob.name}", "wb") as f:#opens a local file for write with path where it'll be saved (uses the same name as blob) and "wb" for binary write mode
        f.write(blob_client.download_blob().readall())#starts download of each Azurite file, reads all content in memory and writes it in the local file
    print(f"Downloaded: {blob.name}")

Downloaded: gdp_raw/part-00000-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00001-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00002-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00003-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00004-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00005-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00006-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet
Downloaded: gdp_raw/part-00007-35724772-3d7d-4527-ae0a-fba2526855b1-c000.snappy.parquet


In [8]:
df = spark.read.parquet("/tmp/bronze/gdp_raw")
df.createOrReplaceTempView("bronze_gdp")
spark.sql("SELECT * FROM bronze_gdp LIMIT 10").show()

+--------------------+---------------+----+-------+--------------------+----------+----+----------------+
|             country|countryiso3code|date|decimal|           indicator|obs_status|unit|           value|
+--------------------+---------------+----+-------+--------------------+----------+----+----------------+
|{value -> Uruguay...|            URY|1992|      1|{value -> GDP per...|          |    |4101.71054155291|
|{value -> Uruguay...|            URY|1991|      1|{value -> GDP per...|          |    |3589.01541942634|
|{value -> Uruguay...|            URY|1990|      1|{value -> GDP per...|          |    |2995.36105663465|
|{value -> Uruguay...|            URY|1989|      1|{value -> GDP per...|          |    |2734.36623871535|
|{value -> Uruguay...|            URY|1988|      1|{value -> GDP per...|          |    |2677.19777690993|
|{value -> Uruguay...|            URY|1987|      1|{value -> GDP per...|          |    | 2415.6541994875|
|{value -> Uruguay...|            URY|1986|   